In [6]:
import numpy as np
import pandas as pd
from pathlib import Path

import os
import random
import shutil
from sklearn.model_selection import train_test_split

In [14]:
PATH_ = Path('keypoints_data/mediapipe/train/Swing')

for npy_file in PATH_.glob('*.npy'):
    array = np.load(npy_file)
    #print('shape:', array.shape)
    #print(np.min(array), np.max(array))

zero_chunks = 0
total_chunks = 0

for npy_file in PATH_.glob('*.npy'):

    array = np.load(npy_file)

    total_chunks += 1

    if np.all(array == 0):
        zero_chunks += 1

print(f'Total chunks: {total_chunks}')
print(f'Zero chunks: {zero_chunks}')
print(f'Percent: {100 * zero_chunks / total_chunks:.2f}%')

Total chunks: 350
Zero chunks: 25
Percent: 7.14%


In [21]:
PATH_ = Path('keypoints_data/mediapipe/test/TableTennisShot')

for npy_file in PATH_.glob('*.npy'):
    array = np.load(npy_file)
    #print('shape:', array.shape)
    #print(np.min(array), np.max(array))


zero_chunks = 0
total_chunks = 0

for npy_file in PATH_.glob('*.npy'):

    array = np.load(npy_file)

    total_chunks += 1

    if np.all(array == 0):
        zero_chunks += 1

print(f'Total chunks: {total_chunks}')
print(f'Zero chunks: {zero_chunks}')
print(f'Percent: {100 * zero_chunks / total_chunks:.2f}%')

Total chunks: 67
Zero chunks: 3
Percent: 4.48%


In [2]:
SOURCE_DIR = "data/Kinetics/train"
OUTPUT_DIR = "data/Kinetics_split"

TRAIN_SIZE = 0.7
VAL_SIZE = 0.15
TEST_SIZE = 0.15

VIDEO_EXTENSIONS = [".avi", ".mp4", ".mov", ".mkv"]
RANDOM_STATE = 42

for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(OUTPUT_DIR, split), exist_ok=True)

def is_video_file(filename):
    return Path(filename).suffix.lower() in VIDEO_EXTENSIONS


def collect_videos(class_dir):
    videos = []

    for file in os.listdir(class_dir):
        if is_video_file(file):
            videos.append(file)

    return videos


def copy_and_collect_metadata(files, class_name, split_name):
    metadata = []

    split_class_dir = os.path.join(OUTPUT_DIR, split_name, class_name)
    os.makedirs(split_class_dir, exist_ok=True)

    source_class_dir = os.path.join(SOURCE_DIR, class_name)

    for file_name in files:
        src_path = os.path.join(source_class_dir, file_name)
        dst_path = os.path.join(split_class_dir, file_name)

        # Копирование файла
        shutil.copy2(src_path, dst_path)

        # clip_name без расширения
        clip_name = Path(file_name).stem

        # относительный путь
        relative_path = f"/{split_name}/{class_name}/{file_name}"

        metadata.append({
            "clip_name": clip_name,
            "clip_path": relative_path,
            "label": class_name
        })

    return metadata



all_train_metadata = []
all_val_metadata = []
all_test_metadata = []

classes = sorted(os.listdir(SOURCE_DIR))

for class_name in classes:

    class_dir = os.path.join(SOURCE_DIR, class_name)

    if not os.path.isdir(class_dir):
        continue

    videos = collect_videos(class_dir)

    if len(videos) == 0:
        continue

    train_files, temp_files = train_test_split(
        videos,
        test_size=(1 - TRAIN_SIZE),
        random_state=RANDOM_STATE
    )

    val_ratio_adjusted = VAL_SIZE / (VAL_SIZE + TEST_SIZE)

    val_files, test_files = train_test_split(
        temp_files,
        test_size=(1 - val_ratio_adjusted),
        random_state=RANDOM_STATE
    )

    
    all_train_metadata.extend(
        copy_and_collect_metadata(train_files, class_name, "train")
    )

    all_val_metadata.extend(
        copy_and_collect_metadata(val_files, class_name, "val")
    )

    all_test_metadata.extend(
        copy_and_collect_metadata(test_files, class_name, "test")
    )

    print(
        f"{class_name}: "
        f"train={len(train_files)}, "
        f"val={len(val_files)}, "
        f"test={len(test_files)}"
    )


train_df = pd.DataFrame(all_train_metadata)
val_df = pd.DataFrame(all_val_metadata)
test_df = pd.DataFrame(all_test_metadata)

train_df.to_csv(
    os.path.join(OUTPUT_DIR, "train.csv"),
    index=False
)

val_df.to_csv(
    os.path.join(OUTPUT_DIR, "val.csv"),
    index=False
)

test_df.to_csv(
    os.path.join(OUTPUT_DIR, "test.csv"),
    index=False
)

print("\nDONE!")
print(f"Dataset saved to: {OUTPUT_DIR}")


abseiling: train=32, val=7, test=7
air drumming: train=31, val=7, test=7
answering questions: train=9, val=2, test=2
applauding: train=6, val=2, test=2
applying cream: train=9, val=2, test=2
archery: train=31, val=7, test=7
arm wrestling: train=30, val=6, test=7
arranging flowers: train=12, val=3, test=3
assembling computer: train=11, val=2, test=3
auctioning: train=10, val=2, test=3
baby waking up: train=13, val=3, test=4
baking cookies: train=22, val=5, test=5
balloon blowing: train=18, val=4, test=5
bandaging: train=12, val=3, test=3
barbequing: train=28, val=6, test=7
bartending: train=13, val=3, test=3
beatboxing: train=23, val=5, test=6
bee keeping: train=8, val=2, test=2
belly dancing: train=25, val=6, test=6
bench pressing: train=30, val=7, test=7
bending back: train=14, val=3, test=4
bending metal: train=7, val=2, test=2
biking through snow: train=29, val=6, test=7
blasting sand: train=18, val=4, test=5
blowing glass: train=30, val=6, test=7
blowing leaves: train=7, val=2, tes

In [13]:
df = pd.read_csv('all_models_full_data.csv')

In [14]:
df

,Unnamed: 0.1,Unnamed: 0,Epoch,Accuracy_train,Accuracy_val,Exp_name,Overfit
0,0.0,NaN,0,0.377823,0.487479,debug_yolo_LSTM_30_naked,-0.109656
1,1.0,NaN,1,0.532426,0.564274,debug_yolo_LSTM_30_naked,-0.031848
2,2.0,NaN,2,0.602779,0.624374,debug_yolo_LSTM_30_naked,-0.021595
3,3.0,NaN,3,0.660973,0.639399,debug_yolo_LSTM_30_naked,0.021574
4,4.0,NaN,4,0.683845,0.659432,debug_yolo_LSTM_30_naked,0.024412
...,...,...,...,...,...,...,...
530,NaN,NaN,10,0.835008,0.785967,debug_yolo_transformer_15_withobject,NaN
531,NaN,NaN,11,0.847568,0.772406,debug_yolo_transformer_15_withobject,NaN
532,NaN,NaN,12,0.846865,0.750590,debug_yolo_transformer_15_withobject,NaN
533,NaN,NaN,13,0.856009,0.778302,debug_yolo_transformer_15_withobject,NaN


In [15]:
df['Exp_name'].unique().item

<bound method ExtensionArray.item of <StringArray>
[                    'debug_yolo_LSTM_30_naked',
                'debug_mediapipe_LSTM_30_naked',
        'debug_yolo_LSTM_30_naked_noskipframes',
         'debug_yolo_GRU_30_naked_noskipframes',
 'debug_yolo_transformer_30_naked_noskipframes',
                    'debug_yolo_LSTM_30_smooth',
                     'debug_yolo_GRU_30_smooth',
             'debug_yolo_transformer_30_smooth',
         'debug_mediapipe_LSTM_30_noskipframes',
          'debug_mediapipe_GRU_30_noskipframes',
  'debug_mediapipe_transformer_30_noskipframes',
            'debug_mediapipe_LSTM_noskipframes',
             'debug_yolo_LSTM_15_withoutObject',
             'debug_yolo_LSTM_25_withoutObject',
              'debug_yolo_GRU_15_withoutObject',
      'debug_yolo_transformer_15_withoutObject',
                'debug_yolo_LSTM_15_withobject',
                 'debug_yolo_GRU_15_withobject',
         'debug_yolo_transformer_15_withobject']
Length: 19, dtype:

In [16]:
df.groupby('Exp_name').max()['Accuracy_val']

Exp_name
debug_mediapipe_GRU_30_noskipframes             0.874580
debug_mediapipe_LSTM_30_naked                   0.869369
debug_mediapipe_LSTM_30_noskipframes            0.871264
debug_mediapipe_LSTM_noskipframes               0.868981
debug_mediapipe_transformer_30_noskipframes     0.857471
debug_yolo_GRU_15_withobject                    0.800708
debug_yolo_GRU_15_withoutObject                 0.740458
debug_yolo_GRU_30_naked_noskipframes            0.880759
debug_yolo_GRU_30_smooth                        0.873646
debug_yolo_LSTM_15_withobject                   0.769458
debug_yolo_LSTM_15_withoutObject                0.772265
debug_yolo_LSTM_25_withoutObject                0.784987
debug_yolo_LSTM_30_naked                        0.864775
debug_yolo_LSTM_30_naked_noskipframes           0.887082
debug_yolo_LSTM_30_smooth                       0.868231
debug_yolo_transformer_15_withobject            0.796580
debug_yolo_transformer_15_withoutObject         0.723919
debug_yolo_transformer

In [17]:
df['Overfit'] = df['Accuracy_train'] - df['Accuracy_val']

In [18]:
df.groupby('Exp_name').max()[['Accuracy_val', 'Overfit']]

,Accuracy_val,Overfit
Exp_name,,
debug_mediapipe_GRU_30_noskipframes,0.874580,0.120258
debug_mediapipe_LSTM_30_naked,0.869369,0.135572
debug_mediapipe_LSTM_30_noskipframes,0.871264,0.132159
debug_mediapipe_LSTM_noskipframes,0.868981,0.132369
debug_mediapipe_transformer_30_noskipframes,0.857471,0.116538
debug_yolo_GRU_15_withobject,0.800708,0.087255
debug_yolo_GRU_15_withoutObject,0.740458,0.070580
debug_yolo_GRU_30_naked_noskipframes,0.880759,0.088612
debug_yolo_GRU_30_smooth,0.873646,0.093170


In [9]:
df.to_csv("all_models_full_data.csv")

In [ ]:
"""
debug_yolo_GRU_15_withobject	0.800708	0.087255
debug_yolo_GRU_15_withoutObject	0.740458	0.070580

debug_yolo_LSTM_15_withobject	    0.769458	0.072766
debug_yolo_LSTM_15_withoutObject	0.772265	0.075741
debug_yolo_LSTM_25_withoutObject	0.784987	0.152742

debug_yolo_transformer_15_withobject	0.796580	0.096275
debug_yolo_transformer_15_withoutObject	0.723919	0.087089

"""